# Piloto CAA · Show-o2 1.5B
Usa GPU con BF16 (A100 o L4) y High-RAM; este piloto carga en BF16. Ejecutar en orden. Extracción: texto → activación de la letra A/B. Evaluación: imagen + prompt → texto.

Adaptación de tu `Experiment_1_Sycophancy.ipynb`. Usamos 20 pares que tú proporcionarás; no reutilizamos sus ejemplos como datos definitivos.

Fuentes: [paper, sección 3](https://arxiv.org/html/2312.06681v4#S3), [CAA](https://github.com/nrimsky/CAA/blob/main/generate_vectors.py), [sampler Show-o2](https://github.com/showlab/Show-o/blob/45a5a2de01d1ebd10cd5864d29310a76476cdf23/show-o2/models/modeling_showo2_qwen2_5.py#L473).


## 1. Preparar una sesión nueva
Este notebook descarga **nuestra rama** y la implementación original de Show-o. No necesita descargar CAA: la adaptación está en `experiments/caa_utils.py`.
La rama debe estar subida a GitHub antes de ejecutar esta celda. Si aún no está publicada, usa las instrucciones locales del README.


In [ ]:
!git clone --branch experiment/caa-understanding --single-branch https://github.com/Nicerova7/show-o2-space.git /content/show-o2-space
!git clone https://github.com/showlab/Show-o.git /content/Show-o
!git -C /content/Show-o checkout 45a5a2de01d1ebd10cd5864d29310a76476cdf23
%pip install -q transformers==4.47.0 diffusers==0.31.0 huggingface-hub==0.27.1 accelerate==1.2.1 timm==1.0.12 einops==0.8.0 torchdiffeq==0.2.5 "numpy>=1.26,<3" "Pillow>=10.4,<12" sentencepiece==0.2.1 pandas


**Reinicia la sesión** después de instalar (Runtime → Restart session). Luego continúa aquí sin repetir la instalación.

In [ ]:
import sys
import json
from pathlib import Path
import torch
import pandas as pd
from PIL import Image
from IPython.display import display
from google.colab import files
from tqdm.auto import tqdm

sys.path[:0] = ["/content/Show-o/show-o2", "/content/show-o2-space"]
from experiments.caa_utils import (
    ShowO2Adapter, MODEL_ID, contrast_tokens,
    capture_answer_letter, steer_understanding,
)

adapter = ShowO2Adapter()
LAYER = 12  # índice de bloque, desde cero; elección piloto, no óptimo validado
COEFFICIENTS = [-2, 0, 2]
MAX_NEW_TOKENS = 100
SEED = 42
OUTPUT = Path("/content/caa_outputs")
OUTPUT.mkdir(exist_ok=True)
print(MODEL_ID, "| capas:", len(adapter.layers), "| GPU:", torch.cuda.get_device_name())


## 2. Pega tus 20 pares
Cada entrada necesita `(pregunta, respuesta con el comportamiento, respuesta opuesta)`. Por ejemplo, si el comportamiento es adulación: la positiva adula y la negativa responde con fundamento. La pregunta y las dos opciones serán iguales dentro de cada par; cambia únicamente la letra elegida.

La lista se deja vacía hasta que pases tus datos. Primero llénala y confirma que el contador muestra 20. La extracción es **textual**, como en tu notebook; la transferencia a imagen + texto se explora después.


In [ ]:
BUILD_EXAMPLES = [
    # ("Tu pregunta", "Respuesta positiva", "Respuesta negativa"),
    # Pega aquí tus 20 tripletas.
]
print("Pares proporcionados:", len(BUILD_EXAMPLES))


In [ ]:
BUILD_TOKENS = [
    contrast_tokens(adapter.tokenizer, question, positive, negative, i)
    for i, (question, positive, negative) in enumerate(BUILD_EXAMPLES)
]

# Auditoría antes de ejecutar el modelo: la última posición debe ser A o B.
token_rows = []
for i, pair in enumerate(BUILD_TOKENS, 1):
    for side, tokens in pair.items():
        token_rows.append({
            "pair": i, "side": side, "position": tokens.shape[1] - 1,
            "token_id": tokens[0, -1].item(),
            "selected_token": adapter.tokenizer.decode(tokens[0, -1:]),
            "last_tokens": [adapter.tokenizer.decode([t]) for t in tokens[0, -8:].tolist()],
        })
audit = pd.DataFrame(token_rows)
display(audit)
print("INPUT POSITIVO COMPLETO:\n", adapter.tokenizer.decode(BUILD_TOKENS[0]["positive"][0]))
audit.to_csv(OUTPUT / "extraction_tokens.csv", index=False)


## 3. Crear el vector mediante resta
El paper extrae la activación **en la letra de respuesta**. El código original usa `-2` porque su respuesta completa es `(A)` o `(B)`. Con Qwen, `(A` se fusiona en un token: por eso aquí la respuesta es una letra suelta `A` o `B` después del encabezado assistant. Verificado con el tokenizer real: el último token es exactamente la letra, y usamos `-1`. No añadimos EOS ni fin de turno después. Revisa arriba que `selected_token` sea A/B.

Capturamos la salida del bloque elegido, antes de la normalización final. Para cada par: `d_i = h_positivo - h_negativo`. Luego: `v = mean(d_i)`. Sin normalización adicional.


In [ ]:
positive_activations = []
negative_activations = []
with torch.inference_mode(), adapter.autocast():
    for pair in tqdm(BUILD_TOKENS):
        with capture_answer_letter(adapter.layers, LAYER) as captured:
            adapter.model.showo(input_ids=pair["positive"].to(adapter.device), use_cache=False)
        positive_activations.append(captured["activation"])
        with capture_answer_letter(adapter.layers, LAYER) as captured:
            adapter.model.showo(input_ids=pair["negative"].to(adapter.device), use_cache=False)
        negative_activations.append(captured["activation"])

h_positive = torch.stack(positive_activations)
h_negative = torch.stack(negative_activations)
differences = h_positive - h_negative
steering_vector = differences.mean(dim=0)

print("Positivas:", h_positive.shape, "| Negativas:", h_negative.shape)
print("Restas:", differences.shape, "| Vector promedio:", steering_vector.shape)
display(pd.DataFrame({
    "h+ (par 1)": h_positive[0, :12].numpy(),
    "h- (par 1)": h_negative[0, :12].numpy(),
    "h+ - h- (par 1)": differences[0, :12].numpy(),
    "v (promedio de pares)": steering_vector[:12].numpy(),
}))
torch.save({"vector": steering_vector, "positive": h_positive, "negative": h_negative,
            "differences": differences, "layer": LAYER, "model": MODEL_ID,
            "build_examples": BUILD_EXAMPLES, "token_audit": token_rows}, OUTPUT / "steering_vector.pt")


## 4. Tu imagen + tu prompt + tres coeficientes
Sube una imagen. Se usa resize + crop central de 432×432 del adapter; mostramos también ese recorte para ver qué recibe el modelo. Cambia el prompt y los tres valores de `COEFFICIENTS` arriba.


In [ ]:
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
TEST_PROMPT = "Describe this image."

from torchvision import transforms
crop = transforms.Compose([
    transforms.Resize(432, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop((432, 432)),
])
display(crop(Image.open(IMAGE_PATH).convert("RGB")))
embeddings, mask, alignment = adapter.prepare_understanding(IMAGE_PATH, TEST_PROMPT)
input_length = embeddings.shape[1]
last_id = alignment["token_ids"][-1]
print("Última posición del input:", input_length - 1)
print("Último token:", repr(adapter.tokenizer.decode([last_id])))
print("Final del input:", adapter.tokenizer.decode(alignment["token_ids"][-12:]))


## 5. Comparar respuestas
`h_nuevo = h + c*v`. Empezamos en `input_length - 1` (incluye el encabezado assistant preparado por Show-o2): esa activación predice el primer token de salida. En cada forward posterior intervenimos esa posición y todas las posiciones de salida ya incorporadas.

El sampler original recalcula toda la secuencia sin KV cache. Por eso el inicio del hook es **fijo**, y su rango crece: 1 posición, luego 2, luego 3… Modificar sólo `[:, -1]` en cada forward perdería la intervención sobre los tokens de salida anteriores. Los embeddings de imagen y el texto anterior al límite no reciben suma directa.

`c=0` es el control, `c>0` apunta al comportamiento positivo y `c<0` al opuesto; el efecto real hay que observarlo. La misma imagen, prompt, vector y seed se usan para los tres valores. Decodificación `top_k=1` del adapter original.


In [ ]:
rows = []
traces = {}
for c in COEFFICIENTS:
    torch.manual_seed(SEED)
    with steer_understanding(adapter.layers, LAYER, steering_vector, c, input_length) as trace:
        response = adapter.answer(embeddings, mask, max_new_tokens=MAX_NEW_TOKENS)
    traces[str(c)] = trace
    rows.append({"image": IMAGE_PATH, "prompt": TEST_PROMPT, "c": c,
                 "layer": LAYER, "seed": SEED, "response": response})
    print(f"\nc = {c}\n{response}")

results = pd.DataFrame(rows)
display(results)
print("Posiciones intervenidas en los primeros forwards:")
display(pd.DataFrame(traces[str(COEFFICIENTS[0])]).head(6))
results.to_csv(OUTPUT / "comparison.csv", index=False)
(OUTPUT / "run.json").write_text(json.dumps({
    "model": MODEL_ID, "layer": LAYER, "coefficients": COEFFICIENTS,
    "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS, "alignment": alignment,
    "trace": traces, "results": rows,
}, indent=2, ensure_ascii=False), encoding="utf-8")


Para otra prueba, cambia la imagen/prompt y repite las secciones 4–5. Para cambiar de capa, repite desde la extracción del vector. Los hooks se retiran al salir de cada contexto.

## 6. Descargar resultados
El ZIP contiene el vector, las activaciones positivas/negativas, las restas, auditoría de tokens, comparación CSV y rangos intervenidos por forward.


In [ ]:
import shutil
shutil.make_archive("/content/caa_outputs", "zip", OUTPUT)
files.download("/content/caa_outputs.zip")
